<a href="https://colab.research.google.com/github/AI-is-out-there/neural-network-skills-review/blob/main/task/task4-1_genom_supervised_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Graph Convolutional Network for Population Classification

This notebook implements a vanilla way GCN application to classify population groups using genetic data
 1. Install required packages
 2. Load and preprocess data
 3. Construct k-NN graph using NetworkX
 4. Train GCN model
 5. Visualize results


Graph Convolutional Networks (GCNs) are particularly well-suited for this genetic population classification task for several compelling reasons that align perfectly with the nature of genetic data and population structure:

### 1. Genetic Data Has Inherent Graph Structure

Human genetic variation naturally forms a graph structure where:
- Individuals are nodes
- Genetic similarity forms edges between nodes
- Populations form natural clusters in this graph

When we constructed our k-NN graph, we observed clear community structure where individuals from the same population (like YRI, LWK, MSL) clustered together. GCNs directly leverage this structure rather than ignoring it (as traditional ML methods would).

### 2. Capturing Population Hierarchy

Human populations have a hierarchical structure that GCNs can effectively model:
- First GCN layer captures immediate genetic neighbors (within-population variation)
- Second GCN layer captures broader population structure (between-population relationships)
- This mirrors real human evolutionary history where populations diverge in a tree-like structure

Traditional classifiers treat all samples as independent, missing these critical hierarchical relationships that define population genetics.

### 3. Information Propagation Through Genetic Lineages

The core operation of GCNs - message passing - directly corresponds to how genetic information flows:
- Each individual's genetic profile is influenced by their ancestors
- GCNs propagate information from genetically similar individuals
- This creates a biologically meaningful smoothing effect where classification decisions incorporate information from related samples

This is particularly valuable for populations with limited samples, as information from genetically similar individuals improves classification accuracy.

### 4. Handling Small Sample Sizes per Population

In population genetics, we often have limited samples per population group (in our case, 7 populations with varying sample sizes). GCNs excel here because:

- They implement a form of semi-supervised learning where unlabeled nodes benefit from labeled neighbors
- Information propagates through the graph, effectively increasing the "effective sample size" for each individual
- This reduces overfitting compared to traditional methods that treat each sample in isolation

### 5. Modeling Linkage Disequilibrium

Genetic variants don't occur independently - nearby SNPs show correlation (linkage disequilibrium). While our current implementation uses a sample graph rather than a feature graph, the GCN approach naturally extends to model both:

- Sample relationships (who is genetically similar to whom)
- Feature relationships (which SNPs tend to co-occur)

This dual perspective is difficult to capture with traditional ML methods but aligns with biological reality.


### 6. Biological Interpretability

GCN representations often align with known population structure:
- The learned embeddings typically preserve geographic and historical relationships
- First GCN layer activations might highlight population-specific variants
- Second layer activations capture broader continental patterns

This interpretability is valuable for biological insights beyond just classification accuracy.

### 7. Practical Advantages for This Dataset

For our specific genetic dataset:
- The binary mutation encoding creates sparse features where traditional methods struggle
- Population boundaries are gradual (clinal variation) rather than discrete
- Genetic similarity forms a non-Euclidean manifold that graphs capture better than Euclidean distance


#Install and Load libraries

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Determine the correct wheel URL based on PyTorch and CUDA versions
TORCH_VERSION = torch.__version__.split('+')[0]

# Format CUDA version correctly (e.g., 'cu118' for CUDA 11.8)
if torch.cuda.is_available():
    CUDA_VERSION = torch.version.cuda.replace('.', '')
    # Format as 'cu11x' where x is the minor version
    CUDA_VERSION = f"cu{CUDA_VERSION[:2]}{CUDA_VERSION[2]}"
else:
    CUDA_VERSION = 'cpu'

print(f"Installing for PyTorch {TORCH_VERSION} and {CUDA_VERSION}")

In [ ]:
# torch-geometric is installed from task/task4_1/requirements.txt.

In [ ]:
# networkx is installed from task/task4_1/requirements.txt.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
torch.manual_seed(37)
np.random.seed(37)

#Load dataset

In [ ]:
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name in {'vanilla-code', 'your-sollution', 'your_solution'}:
    repo_root = repo_root.parent

def dataset_path(task_name: str, filename: str) -> str:
    local_path = repo_root / 'dataset' / task_name / filename
    if local_path.exists():
        return str(local_path)
    return (
        'https://github.com/AI-is-out-there/'
        'check-your-biomed-datascience-skills-review/raw/refs/heads/dev/'
        f'dataset/{task_name}/{filename}'
    )

train_path = dataset_path('task4_1', 'train.txt')
test_features_path = dataset_path('task4_1', 'test_features.txt')

raw_train = pd.read_csv(train_path, sep=r'\s+', header=None)
raw_test = pd.read_csv(test_features_path, sep=r'\s+', header=None)
population = raw_train.iloc[:, 2].astype(str)
nucleotide_codes = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
df = raw_train.iloc[:, 3:].replace(nucleotide_codes).astype(np.float32)
test_X = raw_test.iloc[:, 2:].replace(nucleotide_codes).to_numpy(dtype=np.float32)[:, ::100]
test_sample_ids = raw_test.iloc[:, 0].astype(str).to_numpy()
X = df.to_numpy()[:, ::100]
le = LabelEncoder()
y = le.fit_transform(population)
num_classes = len(le.classes_)


# Visualise dataset

In [ ]:
# Graph Construction (k-NN)

from sklearn.neighbors import kneighbors_graph
import matplotlib.patches as mpatches # Import mpatches for custom legend handles

# Create k-NN graph (k=5)
k = 5
adj_matrix = kneighbors_graph(X, k, mode='connectivity', include_self=False)

# Convert to NetworkX graph
G = nx.from_scipy_sparse_array(adj_matrix)

number_visualise_nodes = 100

# Visualize a subgraph (first 50 nodes) for clarity
plt.figure(figsize=(12, 8))
# Create a subgraph from the first 50 nodes
G_sub = G.subgraph(range(number_visualise_nodes))
# Compute positions only for the nodes in the subgraph
pos = nx.spring_layout(G_sub, seed=37)
nx.draw_networkx_nodes(G_sub, pos, node_size=50,
                       node_color=y[:number_visualise_nodes],
                       cmap='tab10',
                       alpha=0.9)
nx.draw_networkx_edges(G_sub, pos, alpha=0.2)
plt.title('k-NN Graph visualisation', fontsize=15)
plt.axis('off')

# Create legend patches using the same colormap and encoded labels
# le.classes_ and num_classes are available from the previous data loading cell
population_colors = plt.cm.tab10(np.linspace(0, 1, num_classes))
legend_patches = [mpatches.Patch(color=population_colors[i], label=le.classes_[i])
                 for i in range(num_classes)]
plt.legend(handles=legend_patches, title="Populations", loc='best', fontsize=10, title_fontsize=12)

plt.show()

In [ ]:
from scipy.spatial.distance import pdist, squareform
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
# ======================
# Visualize Population Distribution
# ======================
plt.figure(figsize=(13, 5))
population_counts = population.value_counts()
sns.barplot(x=population_counts.index, y=population_counts.values)
plt.title('Population Distribution', fontsize=16)
plt.xlabel('Population', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

# ======================
# Create Genetic Similarity Graph
# ======================
# Convert to numpy array for processing
X = df.values.astype(float)[:, ::100]

# Calculate pairwise genetic distances (Hamming distance)
distances = pdist(X, metric='hamming')
similarity_matrix = 1 - squareform(distances)

# Create k-NN graph (k=5) based on genetic similarity
k = 5
adj_matrix = kneighbors_graph(X, k, mode='connectivity', include_self=False)

# Convert to NetworkX graph
G = nx.from_scipy_sparse_array(adj_matrix)

# Encode populations for coloring
le = LabelEncoder()
pop_encoded = le.fit_transform(population)
pop_classes = le.classes_
num_classes = len(pop_classes)

# Create a color map for populations
population_colors = plt.cm.tab10(np.linspace(0, 1, num_classes))
population_color_dict = {pop: population_colors[i] for i, pop in enumerate(pop_classes)}

# Map colors to nodes
node_colors = [population_color_dict[population[i]] for i in range(len(population))]

# %% [code]
# ======================
# Visualization 1: Force-Directed Graph Layout
# ======================
plt.figure(figsize=(14, 12))

# Use a force-directed layout that emphasizes community structure
pos = nx.spring_layout(G, k=0.3, iterations=100, seed=37)

# Draw the graph
nx.draw_networkx_nodes(
    G, pos,
    node_size=80,
    node_color=node_colors,
    alpha=0.85,
    edgecolors='black',
    linewidths=0.5
)
nx.draw_networkx_edges(
    G, pos,
    width=0.5,
    alpha=0.2,
    edge_color='gray'
)

# Create legend
legend_patches = [mpatches.Patch(color=population_colors[i], label=pop_classes[i])
                 for i in range(num_classes)]
plt.legend(handles=legend_patches, title="Populations",
           loc='best', fontsize=10, title_fontsize=12)

plt.title('Genetic Similarity Network: Population Structure', fontsize=18)
plt.axis('off')
plt.tight_layout()
plt.savefig('genetic_similarity_network.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ======================
# Visualization 2: t-SNE Projection with Network Overlay
# ======================
from sklearn.manifold import TSNE
# Compute t-SNE embedding of the genetic data
tsne = TSNE(n_components=2, random_state=37, perplexity=30)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(14, 12))

# Create a custom colormap with higher contrast
cmap = plt.cm.get_cmap('tab10', num_classes)

# Plot points colored by population
scatter = plt.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=pop_encoded,
    cmap=cmap,
    s=80,
    alpha=0.85,
    edgecolor='black',
    linewidth=0.5,
    zorder=2
)

# Draw edges from the k-NN graph
for edge in G.edges():
    i, j = edge
    plt.plot(
        [X_tsne[i, 0], X_tsne[j, 0]],
        [X_tsne[i, 1], X_tsne[j, 1]],
        'gray', alpha=0.1, linewidth=0.5, zorder=1
    )

# Add population labels for some representative points
for pop in pop_classes:
    idx = np.where(population == pop)[0][0]  # First individual of each population
    plt.annotate(
        pop,
        (X_tsne[idx, 0], X_tsne[idx, 1]),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=12,
        fontweight='bold'
    )

# Add a colorbar/legend
cbar = plt.colorbar(scatter, ticks=range(num_classes))
cbar.set_ticklabels(pop_classes)
cbar.set_label('Population', fontsize=12)

plt.title('t-SNE Projection of Genetic Data with Network Connections', fontsize=18)
plt.xlabel('t-SNE Dimension 1', fontsize=14)
plt.ylabel('t-SNE Dimension 2', fontsize=14)
plt.tight_layout()
plt.savefig('tsne_with_network.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ======================
# Visualization 3: PCA with Population Clusters
# ======================
from sklearn.decomposition import PCA
# Perform PCA on the genetic data
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)

# Create a 3D plot
fig = plt.figure(figsize=(14, 12))
ax = fig.add_subplot(111, projection='3d')

# Plot points colored by population
for i, pop in enumerate(pop_classes):
    idx = np.where(population == pop)[0]
    ax.scatter(
        X_pca[idx, 0], X_pca[idx, 1], X_pca[idx, 2],
        c=[population_colors[i]],
        label=pop,
        s=80,
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )

# Add axis labels and title
ax.set_xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
ax.set_zlabel(f'PCA 3 ({pca.explained_variance_ratio_[2]:.1%})', fontsize=12)
ax.set_title('PCA of Genetic Data Showing Population Clusters', fontsize=16)

# Add legend
ax.legend(title="Populations", loc='best')

# Adjust view angle for better visibility
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig('pca_3d.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ======================
# Visualization 4: Population Network with Edge Weights
# ======================
# Create a population-level network
pop_similarity = np.zeros((num_classes, num_classes))

# Calculate average similarity between populations
for i in range(num_classes):
    for j in range(num_classes):
        pop_i_idx = np.where(population == pop_classes[i])[0]
        pop_j_idx = np.where(population == pop_classes[j])[0]

        # Calculate average similarity between the two populations
        if len(pop_i_idx) > 0 and len(pop_j_idx) > 0:
            pop_similarity[i, j] = np.mean(similarity_matrix[np.ix_(pop_i_idx, pop_j_idx)])

# Create population network
pop_G = nx.Graph()
for i in range(num_classes):
    pop_G.add_node(i, population=pop_classes[i], size=len(np.where(population == pop_classes[i])[0]))

# Add edges with weights based on similarity
for i in range(num_classes):
    for j in range(i+1, num_classes):
        weight = pop_similarity[i, j]
        if weight > 0.52:  # Only show strong connections - Adjusted threshold
            pop_G.add_edge(i, j, weight=weight)

# Create layout for population network
pop_pos = nx.spring_layout(pop_G, seed=42)

plt.figure(figsize=(7, 7))

# Draw nodes with size proportional to population sample size
node_sizes = [pop_G.nodes[i]['size'] * 20 for i in pop_G.nodes()]
nx.draw_networkx_nodes(
    pop_G, pop_pos,
    node_size=node_sizes,
    node_color=[population_colors[i] for i in pop_G.nodes()],
    alpha=0.85,
    edgecolors='black',
    linewidths=1
)

# Draw edges with width proportional to similarity
edge_weights = [pop_G[u][v]['weight'] * 5 for u, v in pop_G.edges()]
nx.draw_networkx_edges(
    pop_G, pop_pos,
    width=edge_weights,
    alpha=0.7,
    edge_color='gray'
)

# Add labels
node_labels = {i: pop_classes[i] for i in pop_G.nodes()}
nx.draw_networkx_labels(pop_G, pop_pos, labels=node_labels, font_size=12, font_weight='bold')

plt.title('Population-Level Genetic Similarity Network', fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.savefig('population_network.png', dpi=300, bbox_inches='tight')
plt.show()

# Main code

In [ ]:
# ======================
# Prepare PyTorch Geometric Data
# ======================
# Convert to PyTorch Geometric Data format
data = Data(
    x=torch.tensor(X, dtype=torch.float),
    edge_index=torch.tensor(np.array(adj_matrix.nonzero()), dtype=torch.long),
    y=torch.tensor(y, dtype=torch.long)
)

# Create train/test split (80/20) and a validation split (10%) from training data
train_idx, test_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=37,
    stratify=y
)
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.125, # 0.125 * 0.8 = 0.1 of total data
    random_state=37,
    stratify=y[train_idx]
)

# Create masks for PyG
data.train_mask = torch.zeros(len(y), dtype=torch.bool)
data.val_mask = torch.zeros(len(y), dtype=torch.bool)
data.test_mask = torch.zeros(len(y), dtype=torch.bool)
data.train_mask[train_idx] = True
data.val_mask[val_idx] = True
data.test_mask[test_idx] = True

print(f"Training samples: {data.train_mask.sum()}")
print(f"Validation samples: {data.val_mask.sum()}")
print(f"Testing samples: {data.test_mask.sum()}")

# ======================
# GCN Model Definition
# ======================
class GCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes, num_layers=2):
        super(GCN, self).__init__()
        self.num_layers = num_layers
        self.conv_layers = torch.nn.ModuleList()

        # First layer
        self.conv_layers.append(GCNConv(num_node_features, hidden_channels))

        # Hidden layers
        for _ in range(num_layers - 2):
            self.conv_layers.append(GCNConv(hidden_channels, hidden_channels))

        # Output layer
        if num_layers > 1:
            self.conv_layers.append(GCNConv(hidden_channels, num_classes))
        else: # Case for a single layer GCN
            self.conv_layers.append(GCNConv(num_node_features, num_classes))

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        for i, conv in enumerate(self.conv_layers):
            x = conv(x, edge_index)
            if i < len(self.conv_layers) - 1: # Apply activation and dropout to all but the last layer
                x = F.elu(x) # Using ELU activation
                x = F.dropout(x, p=0.5, training=self.training)

        return F.log_softmax(x, dim=1)

# Calculate class weights for imbalanced dataset
class_counts = np.bincount(y)
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * num_classes # Normalize weights
print("Class counts:", class_counts)
print("Class weights (normalized):\n", class_weights)

# Initialize model, optimizer, and criterion
model = GCN(
    num_node_features=X.shape[1],
    hidden_channels=256,
    num_classes=num_classes,
    num_layers=3 # Example: 3-layer GCN
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights) # Apply class weights

# ======================
# Model Training
# ======================
from sklearn.metrics import f1_score

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out[mask].argmax(dim=1)
        true_labels = data.y[mask]

        # Calculate accuracy
        acc = int((pred == true_labels).sum()) / int(mask.sum())

        # Calculate F1 score (macro average for multi-class)
        f1 = f1_score(true_labels.cpu().numpy(), pred.cpu().numpy(), average='macro', zero_division=0)
        return acc, f1

# Training loop
best_val_f1 = 0
train_f1_scores = []
val_f1_scores = []

epochs_without_improvement = 10
patience = 180 # Early stopping patience

print("\nStarting training...")
for epoch in range(1, 251):
    loss = train()
    train_acc, train_f1 = evaluate(data.train_mask)
    val_acc, val_f1 = evaluate(data.val_mask)

    train_f1_scores.append(train_f1)
    val_f1_scores.append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_gcn_model_f1.pth') # Save best model based on F1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 20 == 0 or epochs_without_improvement > patience:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '+
              f'Train Acc: {train_acc:.4f}, Train F1: {train_f1:.4f}, '+
              f'Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}')

    if epochs_without_improvement > patience:
        print(f"Early stopping triggered after {patience} epochs without improvement.")
        break

print(f"\nBest Validation F1 Score: {best_val_f1:.4f}")

# Load best model for final evaluation
model.load_state_dict(torch.load('best_gcn_model_f1.pth'))
model.eval()

#Classification visualisation

In [ ]:
# ======================
# Evaluation & Visualization
# ======================
# Get predictions on the test set
out = model(data)
pred = out.argmax(dim=1).detach().numpy()

# Evaluate on test set
test_acc, test_f1 = evaluate(data.test_mask)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test F1 Score (Macro): {test_f1:.4f}")

# Confusion Matrix
plt.figure(figsize=(7, 7))
cm = confusion_matrix(y[data.test_mask], pred[data.test_mask])
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel('Predicted', fontsize=14)
plt.ylabel('True', fontsize=14)
plt.title('Confusion Matrix (Test Set)', fontsize=16)
plt.tight_layout()
plt.show()

# Classification Report
print("Classification Report (Test Set):")
print(classification_report(
    y[data.test_mask],
    pred[data.test_mask],
    target_names=le.classes_,
    zero_division=0 # Handle cases where no predictions are made for a class
))

# Training Progress (F1 Scores)
plt.figure(figsize=(10, 6))
plt.plot(train_f1_scores, label='Train F1 Score')
plt.plot(val_f1_scores, label='Validation F1 Score')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('F1 Score (Macro)', fontsize=12)
plt.title('Training Progress (F1 Score)', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#Conclusion

The GCN model achieved a Test Accuracy of 78.89% and a Macro-averaged Test F1 Score of 78.85% in classifying population groups from genetic data. The confusion matrix indicates that the model performs well for most populations, with high precision and recall for groups like ASW, GWD, and LWK. However, it shows some difficulty distinguishing between ACB, ESN, and YRI, leading to lower scores for these specific classes. The training progress plots for F1 scores show that the model learned effectively, with validation performance tracking training performance reasonably well, suggesting a good balance without severe overfitting or underfitting. Overall, the GCN demonstrates strong capabilities in identifying population structures from genetic information.